# Cross-Country Comparison and Climate Vulnerability Ranking

## Objective
Synthesize cleaned datasets from Ethiopia, Kenya, Sudan, Tanzania, and Nigeria to support a data-driven climate vulnerability ranking for COP32 preparation.

This notebook starts by loading all cleaned country CSVs and concatenating them into one analysis-ready DataFrame.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (14, 7)

In [ ]:
countries = ["ethiopia", "kenya", "sudan", "tanzania", "nigeria"]
data_dir = Path("data")

frames = []
missing_files = []

for country in countries:
    file_path = data_dir / f"{country}_clean.csv"
    if not file_path.exists():
        missing_files.append(str(file_path))
        continue

    frame = pd.read_csv(file_path)

    # Enforce a consistent Country column even if source files differ in casing.
    frame["Country"] = country.title()

    if "DATE" in frame.columns:
        frame["DATE"] = pd.to_datetime(frame["DATE"], errors="coerce")

    frames.append(frame)

if missing_files:
    missing_list = "\n".join(f"- {path}" for path in missing_files)
    raise FileNotFoundError(
        "All five cleaned country CSVs are required for the comparison notebook.\n"
        f"Missing files:\n{missing_list}"
    )

if not frames:
    raise FileNotFoundError("No cleaned country CSVs were found in data/. Run each country EDA notebook export step first.")

df_all = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(frames)} country files.")
print(f"Combined rows: {len(df_all):,}")
print(f"Countries loaded: {sorted(df_all['Country'].unique().tolist())}")
print(f"Columns: {list(df_all.columns)}")
df_all.head()

In [ ]:
monthly_t2m = (
    df_all.dropna(subset=["DATE", "T2M", "Country"])
    .groupby(["Country", pd.Grouper(key="DATE", freq="ME")], as_index=False)["T2M"]
    .mean()
    .rename(columns={"T2M": "monthly_avg_t2m"})
)

plt.figure(figsize=(14, 7))
sns.lineplot(data=monthly_t2m, x="DATE", y="monthly_avg_t2m", hue="Country", linewidth=2)
plt.title("Monthly Average T2M Comparison (2015-2026)")
plt.xlabel("Month")
plt.ylabel("Temperature (C)")
plt.legend(title="Country")
plt.tight_layout()
plt.show()

t2m_summary = (
    df_all.groupby("Country", as_index=False)["T2M"]
    .agg(mean_t2m="mean", median_t2m="median", std_t2m="std")
    .sort_values("mean_t2m", ascending=False)
)

print("T2M summary by country")
display(t2m_summary)

In [ ]:
prectot_clean = df_all.dropna(subset=["Country", "PRECTOTCORR"]).copy()

plt.figure(figsize=(14, 7))
sns.boxplot(data=prectot_clean, x="Country", y="PRECTOTCORR")
plt.title("PRECTOTCORR Variability by Country")
plt.xlabel("Country")
plt.ylabel("PRECTOTCORR (mm/day)")
plt.tight_layout()
plt.show()

prectot_summary = (
    prectot_clean.groupby("Country", as_index=False)["PRECTOTCORR"]
    .agg(mean_prectot="mean", median_prectot="median", std_prectot="std")
    .sort_values("mean_prectot", ascending=False)
)

print("PRECTOTCORR summary by country")
display(prectot_summary)

In [ ]:
events_df = df_all.copy()

if "YEAR" not in events_df.columns:
    if "DATE" not in events_df.columns:
        raise KeyError("Either YEAR or DATE must be present to compute yearly event metrics.")
    events_df["YEAR"] = pd.to_datetime(events_df["DATE"], errors="coerce").dt.year

# Extreme heat: number of days per year with T2M_MAX > 35C.
extreme_heat = (
    events_df.assign(extreme_heat_day=events_df["T2M_MAX"] > 35)
    .groupby(["Country", "YEAR"], as_index=False)["extreme_heat_day"]
    .sum()
    .rename(columns={"extreme_heat_day": "extreme_heat_days"})
)

# Consecutive dry days: maximum dry-day streak per year where PRECTOTCORR < 1 mm.
dry_df = events_df.dropna(subset=["Country", "YEAR", "PRECTOTCORR", "DATE"]).copy()
dry_df["DATE"] = pd.to_datetime(dry_df["DATE"], errors="coerce")
dry_df = dry_df.sort_values(["Country", "YEAR", "DATE"]) 
dry_df["is_dry_day"] = dry_df["PRECTOTCORR"] < 1

def max_dry_streak(group: pd.DataFrame) -> int:
    flags = group["is_dry_day"].to_numpy()
    max_streak = 0
    current = 0
    for flag in flags:
        if flag:
            current += 1
            if current > max_streak:
                max_streak = current
        else:
            current = 0
    return int(max_streak)

max_consecutive_dry = (
    dry_df.groupby(["Country", "YEAR"], as_index=False)
    .apply(lambda group: pd.Series({"max_consecutive_dry_days": max_dry_streak(group)}), include_groups=False)
)

fig, axes = plt.subplots(2, 1, figsize=(14, 12), sharex=True)

sns.barplot(data=extreme_heat, x="YEAR", y="extreme_heat_days", hue="Country", ax=axes[0])
axes[0].set_title("Extreme Heat Days per Year (T2M_MAX > 35C)")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Extreme heat days")
axes[0].legend(title="Country", bbox_to_anchor=(1.02, 1), loc="upper left")

sns.barplot(data=max_consecutive_dry, x="YEAR", y="max_consecutive_dry_days", hue="Country", ax=axes[1])
axes[1].set_title("Maximum Consecutive Dry Days per Year (PRECTOTCORR < 1 mm)")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Max dry-day streak")
axes[1].legend(title="Country", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

print("Extreme heat summary (top rows):")
display(extreme_heat.head())
print("Max consecutive dry-day summary (top rows):")
display(max_consecutive_dry.head())

## Extreme Event Frequency Interpretation
After running the charts above, summarize which countries have the highest yearly extreme-heat burden and longest dry-day streaks.

Connect these patterns to climate vulnerability risks such as heat stress, crop-water stress, and drought preparedness needs.

In [ ]:
t2m_test_df = df_all.dropna(subset=["Country", "T2M"]).copy()

country_groups = [
    group["T2M"].to_numpy()
    for _, group in t2m_test_df.groupby("Country")
    if len(group) > 1
]

if len(country_groups) < 2:
    raise ValueError("Need at least two countries with T2M values for statistical testing.")

anova_stat, anova_p = stats.f_oneway(*country_groups)
kruskal_stat, kruskal_p = stats.kruskal(*country_groups)

tests = pd.DataFrame(
    {
        "test": ["One-way ANOVA", "Kruskal-Wallis"],
        "statistic": [anova_stat, kruskal_stat],
        "p_value": [anova_p, kruskal_p],
    }
)
tests["significant_at_0_05"] = tests["p_value"] < 0.05

display(tests)

for _, row in tests.iterrows():
    conclusion = "significant differences" if row["significant_at_0_05"] else "no statistically significant differences"
    print(f"{row['test']}: p = {row['p_value']:.4g} -> {conclusion} in T2M across countries (alpha = 0.05).")

## Statistical Testing Interpretation
Briefly note both p-values from the ANOVA and Kruskal-Wallis outputs above.

If p < 0.05, conclude that T2M differences across countries are statistically significant; otherwise, note that the observed differences may reflect random variation.

In [ ]:
rank_df = df_all.copy()
rank_df["DATE"] = pd.to_datetime(rank_df["DATE"], errors="coerce")

if "YEAR" not in rank_df.columns:
    rank_df["YEAR"] = rank_df["DATE"].dt.year

# 1) Warming trend: slope of monthly mean T2M over time for each country.
monthly_country_t2m = (
    rank_df.dropna(subset=["Country", "DATE", "T2M"])
    .groupby(["Country", pd.Grouper(key="DATE", freq="ME")], as_index=False)["T2M"]
    .mean()
)
monthly_country_t2m["time_index"] = monthly_country_t2m.groupby("Country").cumcount()

warming_slopes = (
    monthly_country_t2m.groupby("Country")
    .apply(
        lambda g: np.polyfit(g["time_index"], g["T2M"], 1)[0] if len(g) > 1 else np.nan,
        include_groups=False,
    )
    .rename("warming_slope")
    .reset_index()
)

# 2) Precipitation variability: std of PRECTOTCORR.
precip_variability = (
    rank_df.groupby("Country", as_index=False)["PRECTOTCORR"]
    .std()
    .rename(columns={"PRECTOTCORR": "precip_std"})
)

# 3) Extreme heat frequency: yearly mean count of days with T2M_MAX > 35C.
heat_metric = (
    rank_df.assign(extreme_heat_day=rank_df["T2M_MAX"] > 35)
    .groupby(["Country", "YEAR"], as_index=False)["extreme_heat_day"]
    .sum()
    .groupby("Country", as_index=False)["extreme_heat_day"]
    .mean()
    .rename(columns={"extreme_heat_day": "avg_extreme_heat_days"})
)

# 4) Dryness intensity: yearly mean of max consecutive dry-day streak (PRECTOTCORR < 1 mm).
dry_base = rank_df.dropna(subset=["Country", "YEAR", "DATE", "PRECTOTCORR"]).copy()
dry_base = dry_base.sort_values(["Country", "YEAR", "DATE"])
dry_base["is_dry_day"] = dry_base["PRECTOTCORR"] < 1

def max_dry_streak(group: pd.DataFrame) -> int:
    flags = group["is_dry_day"].to_numpy()
    max_streak = 0
    current = 0
    for flag in flags:
        if flag:
            current += 1
            max_streak = max(max_streak, current)
        else:
            current = 0
    return int(max_streak)

dry_streak_metric = (
    dry_base.groupby(["Country", "YEAR"], as_index=False)
    .apply(lambda g: pd.Series({"max_dry_streak": max_dry_streak(g)}), include_groups=False)
    .groupby("Country", as_index=False)["max_dry_streak"]
    .mean()
    .rename(columns={"max_dry_streak": "avg_max_dry_streak"})
)

vulnerability = warming_slopes.merge(precip_variability, on="Country", how="outer")
vulnerability = vulnerability.merge(heat_metric, on="Country", how="outer")
vulnerability = vulnerability.merge(dry_streak_metric, on="Country", how="outer")

metric_cols = ["warming_slope", "precip_std", "avg_extreme_heat_days", "avg_max_dry_streak"]
for col in metric_cols:
    col_min = vulnerability[col].min()
    col_max = vulnerability[col].max()
    if pd.isna(col_min) or pd.isna(col_max) or col_max == col_min:
        vulnerability[f"{col}_norm"] = 0.0
    else:
        vulnerability[f"{col}_norm"] = (vulnerability[col] - col_min) / (col_max - col_min)

vulnerability["vulnerability_score"] = (
    0.30 * vulnerability["warming_slope_norm"]
    + 0.25 * vulnerability["precip_std_norm"]
    + 0.25 * vulnerability["avg_extreme_heat_days_norm"]
    + 0.20 * vulnerability["avg_max_dry_streak_norm"]
)

vulnerability_ranking = vulnerability.sort_values("vulnerability_score", ascending=False).reset_index(drop=True)
vulnerability_ranking["rank"] = vulnerability_ranking.index + 1

display_cols = [
    "rank",
    "Country",
    "vulnerability_score",
    "warming_slope",
    "precip_std",
    "avg_extreme_heat_days",
    "avg_max_dry_streak",
]

print("Climate vulnerability ranking")
display(vulnerability_ranking[display_cols])

## Key Observations for COP32
Use the ranking table and charts above to write a short policy-facing narrative in exactly five bullets:

- Which country is warming fastest and what does the trend suggest?
- Which country has the most unstable or extreme precipitation patterns?
- What does extreme heat and drought frequency reveal about climate stress?
- How does Ethiopia's climate profile compare to its neighbors?
- Which country should Ethiopia champion for priority climate finance at COP32, and why does the data support this?

## Vulnerability Ranking
Build a transparent ranking using evidence from warming trend, precipitation variability, and extreme-event frequency.

The score below uses normalized indicators and a weighted composite:
- Warming trend slope (30%)
- Precipitation variability (25%)
- Extreme heat frequency (25%)
- Consecutive dry-day intensity (20%)

## Comparison Outputs Added
The notebook now includes cross-country comparisons for both temperature trends and precipitation variability, each with a chart and a summary statistics table.

## Next Step
Define a vulnerability score from selected climate-risk indicators and produce a ranked country list for the COP32 position paper.